# When a blurry view still answers

A query that already depends only on the samples you hold *does* factor through those samples. Parity of an integer is a blurry view of the integer. A query that reads only evenness is constant on that blur, so the calculator can answer from the blur. A query that reads the integer itself is not constant on the blur, and the calculator refuses.

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_cs_q_bridge_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_cs_q_bridge_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_cs_q_bridge_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Rational
from coda_cs_q_bridge_core import (
    CofinalShadowTower,
    NamedTerminalizers,
    ProfileFactorizationSolver,
    QueryProbe,
    ShadowNode,
    reproduce_section_0_worked_example,
    terminals_agree,
)

## Evenness is enough for a parity query

The profile is $n\bmod 2$. The query sends evens to $10$ and odds to $20$. Every even looks the same, and the query agrees on them. Theorem CSQ-T1 builds an image factor. The identity query splits the even fibre, so it does not factor.

In [2]:
nodes = [ShadowNode("parity", 1)]
projectors = {"parity": lambda n: n % 2}
tower = CofinalShadowTower(nodes, projectors, {})
domain = [2, 4, 6, 1, 3, 5]

q_parity = QueryProbe(
    "parity_label",
    degree=1,
    eval_fn=lambda n: Rational(10) if n % 2 == 0 else Rational(20),
)
ok, q_bar, err = ProfileFactorizationSolver(tower, q_parity, domain).solve_factorization()
print("parity query factors?", ok)
print("even fibre answers:", q_bar((0,)) if q_bar else None)
print("odd fibre answers:", q_bar((1,)) if q_bar else None)
assert ok is True
assert q_bar((0,)) == Rational(10)
assert q_bar((1,)) == Rational(20)
assert err is None

q_id = QueryProbe("identity", degree=1, eval_fn=lambda n: Rational(n))
ok_id, q_bar_id, err_id = ProfileFactorizationSolver(tower, q_id, domain).solve_factorization()
print("identity query factors?", ok_id)
assert ok_id is False
assert q_bar_id is None
assert "Fiber constancy violation" in str(err_id)

parity query factors? True
even fibre answers: 10
odd fibre answers: 20
identity query factors? False


## The honest reverse is a value plus a witness

An `ImagePoint` is a profile value together with a state that produced it. Evaluating the query at that witness is well-defined because the query is fibre-constant. Asking the factor at a profile that never occurred is the unused-label refusal of Theorem CSQ-T5.

In [3]:
solver = ProfileFactorizationSolver(tower, q_parity, domain)
ok, q_bar, _ = solver.solve_factorization()
pts = solver.construct_image_points()
print("how many witnesses:", len(pts))
for pt in pts[:2]:
    print("val", pt.val, "witness", pt.witness, "answer", pt.evaluate_factored(q_parity, q_bar))
    assert pt.evaluate_factored(q_parity, q_bar) == q_parity.evaluate(pt.witness)
    assert q_bar(pt.val) == q_parity.evaluate(pt.witness)
try:
    q_bar((99,))
    raise AssertionError("unused label should refuse")
except KeyError as exc:
    print("unused label:", exc)

named = NamedTerminalizers(
    tau_q=q_parity.evaluate,
    tau_coda_cs=lambda n: q_bar((n % 2,)),
)
agree, mismatch = terminals_agree(domain, named.tau_q, named.tau_coda_cs)
print("named maps agree?", agree)
assert agree is True
assert mismatch is None

how many witnesses: 6
val (0,) witness 2 answer 10
val (0,) witness 4 answer 10
unused label: 'Profile (99,) is outside the realized domain image (total reverse is OPEN without section/choice, per CSQ-T5)'
named maps agree? True


## A grid energy is a query of the samples

On $\{0,1/4,1/2,3/4\}$ with $X(t)=2t$ at those nodes, the shipped quadrature returns $7/8$. That number is a function of the four samples, so two copies with the same samples get the same answer.

In [4]:
replay = reproduce_section_0_worked_example()
print("energy:", replay["energy_exact"])
print("same samples, same answer?", replay["is_admissible"])
assert replay["energy_exact"] == Rational(7, 8)
assert replay["is_admissible"] is True

energy: 7/8
same samples, same answer? True
